In [2]:
# ═══════════════════════════════════════════════════════════════
# CELL 1: Setup & Imports
# ═══════════════════════════════════════════════════════════════
import sys, os, time, warnings
warnings.filterwarnings('ignore')

# Thêm src vào path
possible_src_paths = [
    '/home/jovyan/work/src',
    os.path.abspath('../src'),
    os.path.abspath('./src')
]
for path in possible_src_paths:
    if os.path.exists(path):
        if path not in sys.path:
            sys.path.insert(0, path)
        print(f"✅ Loaded src from: {path}")
        break

from config.spark_config import create_spark_session, bronze_path, silver_path, gold_path
from config.minio_config import ensure_buckets_exist, get_minio_client

print("✅ Imports OK")

✅ Loaded src from: /home/jovyan/work/src
✅ Imports OK


In [3]:
# ═══════════════════════════════════════════════════════════════
# CELL 2: Khởi tạo Spark + MinIO buckets
# ═══════════════════════════════════════════════════════════════
# Đảm bảo 3 buckets tồn tại

ensure_buckets_exist()


# Tạo SparkSession (lần đầu ~2-3 phút download JARs)
spark = create_spark_session('BronzeIngestion')

print(f"\n{'='*60}")
print(f"Spark version:  {spark.version}")
print(f"Spark UI:       http://localhost:4040")
print(f"Bronze bucket:  {bronze_path()}")
print(f"{'='*60}")

Bucket exists: electronics-bronze
Bucket exists: electronics-silver
Bucket exists: electronics-gold
SparkSession created: BronzeIngestion
MinIO endpoint: http://minio:9000

Spark version:  3.5.0
Spark UI:       http://localhost:4040
Bronze bucket:  s3a://electronics-bronze


In [4]:
# ═══════════════════════════════════════════════════════════════
# CELL 3: Định nghĩa Schema tường minh (tránh schema drift)
# MAP PHASE: Spark sẽ dùng schema này để parse từng JSON record
# ═══════════════════════════════════════════════════════════════
from pyspark.sql.types import (
    StructType, StructField,
    StringType, FloatType, LongType,
    IntegerType, BooleanType, ArrayType, MapType
)

# Schema Review file — 10 fields (đã verify trong 00-source-analyst.ipynb)
REVIEW_SCHEMA = StructType([
    StructField("rating",            FloatType(),   nullable=True),
    StructField("title",             StringType(),  nullable=True),
    StructField("text",              StringType(),  nullable=True),
    StructField("images",            ArrayType(MapType(StringType(), StringType())), nullable=True),
    StructField("asin",              StringType(),  nullable=False),
    StructField("parent_asin",       StringType(),  nullable=False),
    StructField("user_id",           StringType(),  nullable=False),
    StructField("timestamp",         LongType(),    nullable=True),  # Unix ms
    StructField("helpful_vote",      IntegerType(), nullable=True),
    StructField("verified_purchase", BooleanType(), nullable=True),
])

# Schema Metadata file — 14 fields (đã verify trong source analyst)
META_SCHEMA = StructType([
    StructField("main_category",   StringType(),  nullable=True),
    StructField("title",           StringType(),  nullable=True),
    StructField("average_rating",  FloatType(),   nullable=True),
    StructField("rating_number",   IntegerType(), nullable=True),
    StructField("features",        StringType(),  nullable=True),  # JSON string
    StructField("description",     StringType(),  nullable=True),  # JSON string
    StructField("price",           StringType(),  nullable=True),
    StructField("images",          StringType(),  nullable=True),  # JSON string
    StructField("videos",          StringType(),  nullable=True),  # JSON string
    StructField("store",           StringType(),  nullable=True),
    StructField("categories",      StringType(),  nullable=True),  # JSON string
    StructField("details",         StringType(),  nullable=True),  # JSON string
    StructField("parent_asin",     StringType(),  nullable=False),
    StructField("bought_together", StringType(),  nullable=True),  # JSON string
])

print("✅ Schemas defined:")
print(f"   Review schema:  {len(REVIEW_SCHEMA.fields)} fields")
print(f"   Metadata schema: {len(META_SCHEMA.fields)} fields")

✅ Schemas defined:
   Review schema:  10 fields
   Metadata schema: 14 fields


In [5]:
# ═══════════════════════════════════════════════════════════════
# CELL 4: [MapReduce] Stream từ HuggingFace → Clean → Write Direct to MinIO
# KHÔNG lưu xuống ổ cứng local. Streaming batch process.
# ═══════════════════════════════════════════════════════════════
from datasets import load_dataset
import io
import json
import pandas as pd
from config.spark_config import bronze_path

# Cấu hình Batch size (số lượng dòng xử lý mỗi lần để tránh OOM)
BATCH_SIZE = 100_000 

def ingest_to_minio_direct(dataset_name, config_name, spark_schema, minio_path_func, layer_name="records"):
    """
    Stream data từ HuggingFace, convert sang Spark DataFrame và append vào MinIO
    """
    print(f"\n🚀 Bắt đầu Ingestion: {config_name}")
    print(f"   Target: {minio_path_func(f'{layer_name}')}")
    
    # Load streaming dataset
    ds = load_dataset(
        dataset_name,
        config_name,
        split="full",
        streaming=True,
        trust_remote_code=True
    )
    
    batch = []
    batch_idx = 0
    total_records = 0
    t_start = time.time()
    
    # Target path on MinIO
    target_path = minio_path_func(layer_name)
    
    # Xoá dữ liệu cũ nếu muốn overwrite (cẩn thận)
    # create_minio_client().remove_object(...) -> logic này Spark mode='overwrite' sẽ lo ở batch đầu
    
    write_mode = "overwrite" # Batch đầu tiên overwrite, các batch sau append

    for i, record in enumerate(ds):
        # MAP PHASE (Simple): Chuẩn hoá dữ liệu cơ bản tại chỗ nếu cần
        # Ví dụ: convert array thành string nếu schema yêu cầu, ở đây ta giữ nguyên dict
        batch.append(record)
        
        # Khi đủ batch -> Gửi lên Spark -> MinIO
        if len(batch) >= BATCH_SIZE:
            # Tạo DataFrame từ list of dicts
            # Lưu ý: Convert sang Pandas trước thường nhanh hơn createDataFrame trực tiếp từ list dict lớn
            pdf = pd.DataFrame(batch)
            
            # Spark Create DataFrame
            # Force schema để đảm bảo đúng cấu trúc
            df = spark.createDataFrame(pdf, schema=spark_schema)
            
            # REDUCE/KEY-VALUE STORE PHASE: Save to MinIO (Parquet/JSON)
            # Dùng Parquet cho tối ưu, hoặc JSON nếu muốn giữ raw format
            df.write.mode(write_mode).parquet(target_path)
            
            total_records += len(batch)
            print(f"   ✅ Batch {batch_idx+1}: Saved {len(batch):,} records -> MinIO. Total: {total_records:,}")
            
            # Reset
            batch = []
            batch_idx += 1
            write_mode = "append" # Các batch sau nối đuôi vào
            
            # Clean memory
            del pdf, df
            
    # Xử lý batch cuối cùng (nếu còn dư)
    if batch:
        pdf = pd.DataFrame(batch)
        df = spark.createDataFrame(pdf, schema=spark_schema)
        df.write.mode(write_mode).parquet(target_path)
        total_records += len(batch)
        print(f"   ✅ Batch final: Saved {len(batch):,} records -> MinIO.")

    duration = (time.time() - t_start) / 60
    print(f"🏁 Hoàn tất {config_name}. Tổng: {total_records:,} records trong {duration:.1f} phút.")



In [6]:
# ── 1. Ingest Reviews ─────────────────────────────────────────
# Kiểm tra xem dữ liệu đã tồn tại trên MinIO chưa bằng cách thử đọc
try:
    spark.read.parquet(bronze_path('reviews')).limit(1).count()
    print("✅ Bronze Reviews đã tồn tại trên MinIO. Bỏ qua ingestion.")
except:
    print("⏳ Bronze Reviews chưa có. Bắt đầu tải...")
    ingest_to_minio_direct(
        "McAuley-Lab/Amazon-Reviews-2023",
        "raw_review_Electronics",
        REVIEW_SCHEMA,
        bronze_path,
        "reviews"
    )

✅ Bronze Reviews đã tồn tại trên MinIO. Bỏ qua ingestion.


In [7]:
# ── 2. Ingest Metadata ────────────────────────────────────────
try:
    spark.read.parquet(bronze_path('metadata')).limit(1).count()
    print("✅ Bronze Metadata đã tồn tại trên MinIO. Bỏ qua ingestion.")
except:
    print("⏳ Bronze Metadata chưa có. Bắt đầu tải...")
    ingest_to_minio_direct(
        "McAuley-Lab/Amazon-Reviews-2023",
        "raw_meta_Electronics",
        META_SCHEMA,
        bronze_path,
        "metadata"
    )

✅ Bronze Metadata đã tồn tại trên MinIO. Bỏ qua ingestion.


In [9]:
df = spark.read.parquet(bronze_path("reviews"))


In [10]:
df.printSchema()


root
 |-- rating: float (nullable = true)
 |-- title: string (nullable = true)
 |-- text: string (nullable = true)
 |-- images: array (nullable = true)
 |    |-- element: map (containsNull = true)
 |    |    |-- key: string
 |    |    |-- value: string (valueContainsNull = true)
 |-- asin: string (nullable = true)
 |-- parent_asin: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- helpful_vote: integer (nullable = true)
 |-- verified_purchase: boolean (nullable = true)



In [13]:
# Lấy 100 dòng mẫu
pdf = df.limit(10).toPandas()

display(pdf)  



,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,5.0,Great value,Made very well so it gives you great peace of ...,[],B0BYYKGT7X,B0CGHVQ4M3,AGSPIUWZ6JJ6N2HKLHGV7VBSN6HA_1,1683426832279,0,False
1,5.0,Portable monitor,This portable monitor has great picture qualit...,[],B0BBNCFJG1,B0CF21CMG3,AGSPIUWZ6JJ6N2HKLHGV7VBSN6HA_1,1683425287723,0,False
2,5.0,Digital Picture Frame,This digital picture frame is great to have at...,[],B0BNPWB2JD,B0BZSYPRRH,AGSPIUWZ6JJ6N2HKLHGV7VBSN6HA_1,1682885151591,0,False
3,5.0,Philips TV,I really love this TV specialist Phillips. It'...,[],B0BKTN5QXM,B0BKTM9JCF,AGSPIUWZ6JJ6N2HKLHGV7VBSN6HA_1,1681860623178,0,False
4,5.0,Lightning Cable,This lighting cable is super cute and fun. It ...,[],B096BMPLJH,B0C998B965,AGSPIUWZ6JJ6N2HKLHGV7VBSN6HA_1,1681860179149,0,False
5,5.0,HDMI splitter,I really like this HDMI splitter! Even has a r...,[],B0BFPWF6T7,B0BFPWF6T7,AGSPIUWZ6JJ6N2HKLHGV7VBSN6HA_1,1681858233638,0,False
6,5.0,Surround sound bar,This sound bar is really perfect for our movie...,[],B0BCGX4GBG,B0BCGX4GBG,AGSPIUWZ6JJ6N2HKLHGV7VBSN6HA_1,1681831929357,1,False
7,5.0,Headset,"This product is very easy to use, instruction ...",[],B09VYH5C9D,B0BSRGYNWV,AGSPIUWZ6JJ6N2HKLHGV7VBSN6HA_1,1680375157911,0,False
8,5.0,Love it!,"This product is very easy to use, instruction ...",[],B086ZNFPJ4,B0C1GY5C2T,AGSPIUWZ6JJ6N2HKLHGV7VBSN6HA_1,1680374511720,5,False
9,5.0,Bluetooth speaker,"This is very cool to have, a Bluetooth ceiling...",[],B09Y24RNYD,B09Y24RNYD,AGSPIUWZ6JJ6N2HKLHGV7VBSN6HA_1,1680374009577,0,False
